# Financial Analysis

In [174]:
# imports
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt
import numpy as np

In [175]:
# loading data
gates = pd.read_csv("../data/processed/gates_clean.csv")

In [176]:
flight_costs = pd.read_csv("../data/processed/flight_costs_df.csv")

In [177]:
bookings = pd.read_csv("../data/processed/bookings.csv")

In [178]:
bookings_routes = pd.read_csv("../data/processed/booking_routes.csv")

In [179]:
gates.head(5)

,City,Gate_Number,Route_ID,Year,Annual_Gate_Cost
0,Abbotsford,G1,YXX-YYC,2024-2025,166800
1,Calgary,B1,YYC-YVR,2024-2025,373200
2,Calgary,B1,YYC-YYZ,2024-2025,373200
3,Calgary,B1,YYC-YEG,2024-2025,373200
4,Calgary,B2,YYC-YWG,2024-2025,351900


In [180]:
flight_costs.head(5)

,Cost_Date,Route_ID,From,To,Crew,Food,Fuel,Maintenance,Total
0,2024-09-01 06:30:00,YVR-YLW,Vancouver,Kelowna,1388.53,170.52,1401.75,1657.45,4618.25
1,2024-09-01 06:35:00,YLW-YYC,Kelowna,Calgary,1673.28,221.60,1800.44,1898.61,5593.93
2,2024-09-01 06:45:00,YYZ-YYC,Toronto,Calgary,12105.73,2037.11,16016.15,9391.66,39550.65
3,2024-09-01 06:50:00,YLW-YVR,Kelowna,Vancouver,1458.14,187.03,1532.94,1540.96,4719.07
4,2024-09-01 06:50:00,YQR-YYC,Regina,Calgary,2745.73,333.72,2923.61,2504.61,8507.67


In [181]:
bookings.head(9)

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
0,BK100000,2026,0.0,0.0,0.00,0.0,0.0,683.89,683.89,1,Direct
1,BK100001,2025,0.0,0.0,27.81,0.0,0.0,558.37,586.18,1,Direct
2,BK100002,2025,0.0,0.0,0.00,0.0,0.0,796.89,796.89,1,Direct
3,BK100003,2025,0.0,0.0,0.00,0.0,0.0,596.12,596.12,1,Direct
4,BK100004,2025,0.0,0.0,0.00,0.0,0.0,743.32,743.32,1,Direct
5,BK100005,2026,0.0,0.0,0.00,0.0,0.0,1395.40,1395.40,1,Direct
6,BK100006,2025,0.0,0.0,0.00,0.0,0.0,1339.30,1339.30,2,Connecting
7,BK100007,2025,0.0,0.0,0.00,0.0,0.0,865.09,865.09,1,Direct
8,BK100008,2026,0.0,0.0,0.00,0.0,0.0,562.55,562.55,1,Direct


In [182]:
bookings_routes[bookings_routes["Booking_ID"].duplicated(keep = False)]

,Booking_ID,Route_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total
6,BK100006,YWG-YYC,2025,0.0,0.00,0.00,0.0,0.0,824.53,824.53
7,BK100006,YYC-YVR,2025,0.0,0.00,0.00,0.0,0.0,514.77,514.77
11,BK100010,YVR-YYC,2026,0.0,0.00,0.00,0.0,0.0,834.79,834.79
12,BK100010,YYC-YWG,2026,0.0,0.00,0.00,0.0,0.0,1107.79,1107.79
39,BK100037,YVR-YYZ,2025,0.0,0.00,7.23,0.0,0.0,1811.96,1819.19
...,...,...,...,...,...,...,...,...,...,...
370922,BK452950,YYZ-YHZ,2025,0.0,0.00,0.00,0.0,0.0,752.51,752.51
370949,BK452977,YYC-YQR,2025,0.0,0.00,0.00,0.0,0.0,675.88,675.88
370950,BK452977,YYC-YQR,2026,0.0,39.48,0.00,0.0,0.0,0.00,39.48
370965,BK452992,YVR-YYZ,2025,0.0,0.00,0.00,0.0,0.0,1545.27,1545.27


## Gate Lease Preparation

In [183]:
# --- 2024-2025 LEASE YEAR ---

# Filter gates leased during January-December 2024
gates_2425 = gates[gates["Year"] == "2024-2025"].copy()

# Remove duplicate gates that are shared across multiple routes
gates_2425 = gates_2425.drop_duplicates(
    subset=["City", "Gate_Number"]
)

# Calculate annual gate costs for each airport
airport_costs_2425 = (
    gates_2425
    .groupby("City")["Annual_Gate_Cost"]
    .sum()
    .reset_index()
)


# --- 2025-2026 LEASE YEAR ---

# Filter gates leased during January-December 2025
gates_2526 = gates[gates["Year"] == "2025-2026"].copy()

# Remove duplicate gates shared across multiple routes
gates_2526 = gates_2526.drop_duplicates(
    subset=["City", "Gate_Number"]
)

# Calculate annual gate costs for each airport
airport_costs_2526 = (
    gates_2526
    .groupby("City")["Annual_Gate_Cost"]
    .sum()
    .reset_index()
)


# --- PREPARE DATA FOR FISCAL YEAR CALCULATIONS ---

# Merge the lease-year costs so each airport has costs for both years
airport_costs = airport_costs_2425.merge(
    airport_costs_2526,
    on="City",
    suffixes=("_2425", "_2526"),
    validate="one_to_one"
)

# Calculate the percentage change in gate costs at each airport
airport_costs["Growth_Rate"] = (
    airport_costs["Annual_Gate_Cost_2526"]
    / airport_costs["Annual_Gate_Cost_2425"] - 1
)

# Estimate missing 2026-2027 lease costs
# Assume each airport's annual percentage change continues
airport_costs["Estimated_2627"] = (
    airport_costs["Annual_Gate_Cost_2526"]
    * (1 + airport_costs["Growth_Rate"])
)

# Display the prepared data
display(airport_costs.head())

,City,Annual_Gate_Cost_2425,Annual_Gate_Cost_2526,Growth_Rate,Estimated_2627
0,Abbotsford,166800,170200,0.020384,1.736693e+05
1,Calgary,1065800,1090600,0.023269,1.115977e+06
2,Edmonton,490700,500800,0.020583,5.111079e+05
3,Halifax,700200,714800,0.020851,7.297044e+05
4,Kelowna,203800,207200,0.016683,2.106567e+05


## Fiscal Year 2024-2025

### Firm Costs

#### Gates

In [184]:
# FY2024-25: Sep-Dec 2024 + Jan-Aug 2025
gate_total_fy_2425 = (
    airport_costs["Annual_Gate_Cost_2425"].sum() * (4/12)
    + airport_costs["Annual_Gate_Cost_2526"].sum() * (8/12)
)

print(f"FY2024-25 gate costs: ${gate_total_fy_2425:,.2f}")

FY2024-25 gate costs: $9,806,600.00


#### Flights

In [185]:
flight_costs.dtypes

Cost_Date          str
Route_ID           str
From               str
To                 str
Crew           float64
Food           float64
Fuel           float64
Maintenance    float64
Total          float64
dtype: object

In [187]:
# Select flights from September 2024 through August 2025
flights_2425 = flight_costs[
    (flight_costs["Cost_Date"] >= "2024-09-01") &
    (flight_costs["Cost_Date"] < "2025-09-01")
]
flights_2425_total_costs = flights_2425["Total"].sum().round(2)
flights_2425_total_crew = flights_2425["Crew"].sum().round(2)
flights_2425_total_fuel = flights_2425["Fuel"].sum().round(2)
flights_2425_total_food = flights_2425["Food"].sum().round(2)
flights_2425_total_maintenance = flights_2425["Maintenance"].sum().round(2)

print(f"Total costs for 2024-2025 were: ${flights_2425_total_costs}")
print(f"Total crew costs for 2024-2025 were: ${flights_2425_total_crew}")
print(f"Total fuel costs for 2024-2025 were: ${flights_2425_total_fuel}")
print(f"Total food costs for 2024-2025 were: ${flights_2425_total_food}")
print(f"Total maintenance costs for 2024-2025 were: ${flights_2425_total_maintenance}")


Total costs for 2024-2025 were: $192985563.99
Total crew costs for 2024-2025 were: $59717729.49
Total fuel costs for 2024-2025 were: $67061083.08
Total food costs for 2024-2025 were: $7822627.71
Total maintenance costs for 2024-2025 were: $58384123.71


#### All Costs

In [189]:
total_costs_2425 = gate_total_fy_2425 + flights_2425_total_costs
print(f"Total costs for the fiscal year 2024-2025: ${total_costs_2425}")

Total costs for the fiscal year 2024-2025: $202792163.99


### Firm Revenue

In [190]:
bookings.head(10)

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
0,BK100000,2026,0.0,0.0,0.00,0.0,0.0,683.89,683.89,1,Direct
1,BK100001,2025,0.0,0.0,27.81,0.0,0.0,558.37,586.18,1,Direct
2,BK100002,2025,0.0,0.0,0.00,0.0,0.0,796.89,796.89,1,Direct
3,BK100003,2025,0.0,0.0,0.00,0.0,0.0,596.12,596.12,1,Direct
4,BK100004,2025,0.0,0.0,0.00,0.0,0.0,743.32,743.32,1,Direct
5,BK100005,2026,0.0,0.0,0.00,0.0,0.0,1395.40,1395.40,1,Direct
6,BK100006,2025,0.0,0.0,0.00,0.0,0.0,1339.30,1339.30,2,Connecting
7,BK100007,2025,0.0,0.0,0.00,0.0,0.0,865.09,865.09,1,Direct
8,BK100008,2026,0.0,0.0,0.00,0.0,0.0,562.55,562.55,1,Direct
9,BK100009,2025,0.0,0.0,0.00,0.0,0.0,934.58,934.58,1,Direct


In [191]:
bookings_2425 = bookings[bookings["Fiscal_Year"] == 2025]
bookings_2425.head(10)

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
1,BK100001,2025,0.0,0.0,27.81,0.00,0.0,558.37,586.18,1,Direct
2,BK100002,2025,0.0,0.0,0.00,0.00,0.0,796.89,796.89,1,Direct
3,BK100003,2025,0.0,0.0,0.00,0.00,0.0,596.12,596.12,1,Direct
4,BK100004,2025,0.0,0.0,0.00,0.00,0.0,743.32,743.32,1,Direct
6,BK100006,2025,0.0,0.0,0.00,0.00,0.0,1339.30,1339.30,2,Connecting
7,BK100007,2025,0.0,0.0,0.00,0.00,0.0,865.09,865.09,1,Direct
9,BK100009,2025,0.0,0.0,0.00,0.00,0.0,934.58,934.58,1,Direct
11,BK100011,2025,0.0,0.0,28.08,0.00,0.0,1364.45,1392.53,1,Direct
13,BK100013,2025,0.0,0.0,0.00,0.00,0.0,1150.43,1150.43,1,Direct
14,BK100014,2025,0.0,0.0,0.00,-1159.95,0.0,1159.95,0.00,1,Direct


In [192]:
# 2024-2025 revenue streams

# total 
bookings_total_cost_2425 = bookings_2425["Total"].sum().round(2)
# change fee
change_fee_2425 = bookings_2425["Change Fee"].sum().round(2)
# extra baggage
baggage_2425 = bookings_2425["Extra Baggage"].sum().round(2)
# food and onboard
foodboard_2425 = bookings_2425["Food & Onboard"].sum().round(2)
# refund
refund_2425 = bookings_2425["Refund"].sum().round(2)
# seat upgrades
upgrades_2425 = bookings_2425["Seat Upgrade"].sum().round(2)
# ticket
ticket_2425 = bookings_2425["Ticket"].sum().round(2)

In [253]:
# total
print(f"Total revenue for 2025 fiscal year: ${bookings_total_cost_2425}")
# change
print(f"Total revenue from changing flights for 2025 fiscal year: ${change_fee_2425}")
# extra baggage
print(f"Total revenue from extra baggage for 2025 fiscal year: ${baggage_2425}")
# food
print(f"Total revenue from food for 2025 fiscal year: ${foodboard_2425}")
# refund
print(f"Total refunds for 2025 fiscal year: ${refund_2425}")
# upgrade
print(f"Total revenue from upgrades for 2025 fiscal year: ${upgrades_2425}")
# ticket
print(f"Total revenue from tickets for 2025 fiscal year: ${ticket_2425}")


Total revenue for 2025 fiscal year: $180907988.44
Total revenue from changing flights for 2025 fiscal year: $560250.0
Total revenue from extra baggage for 2025 fiscal year: $2005211.93
Total revenue from food for 2025 fiscal year: $436609.38
Total refunds for 2025 fiscal year: $-6700342.73
Total revenue from upgrades for 2025 fiscal year: $681238.09
Total revenue from tickets for 2025 fiscal year: $183925021.77


##### Connecting vs Direct

In [254]:
bookings_c2425 = bookings_2425[bookings_2425["Booking_Type"] == "Connecting"]
bookings_d2425 = bookings_2425[bookings_2425["Booking_Type"] == "Direct"]

In [255]:
bookings_c2425["Total"].sum()

np.float64(12469492.65)

In [256]:
bookings_d2425["Total"].sum()

np.float64(168438495.79000002)

In [257]:
print(f"There are {bookings_c2425.shape[0]} direct flights and {bookings_d2425.shape[0]} connecting")
prop_c2425 = (bookings_c2425.shape[0] / bookings_2425.shape[0]) * 100
prop_d2425 = (bookings_d2425.shape[0] / bookings_2425.shape[0]) * 100
print(f"Proportion of direct flights: {round(prop_d2425, 2)}%")
print(f"Proportion of connecting flights: {round(prop_c2425, 2)}%")

There are 6384 direct flights and 165722 connecting
Proportion of direct flights: 96.29%
Proportion of connecting flights: 3.71%


In [258]:
revenue_prop_c2425 = (bookings_c2425["Total"].sum() / bookings_2425["Total"].sum()) * 100
revenue_prop_d2425 = (bookings_d2425["Total"].sum() / bookings_2425["Total"].sum()) * 100

print(f"Proportion of revenue from direct flights: {round(revenue_prop_d2425, 2)}%")
print(f"Proportion of revenue connecting flights: {round(revenue_prop_c2425, 2)}%")

Proportion of revenue from direct flights: 93.11%
Proportion of revenue connecting flights: 6.89%


## Fiscal Year 2025-2026

### Firm Costs

#### Gates

In [259]:
# FY2025-26: Sep-Dec 2025 + Jan-Aug 2026
gate_total_fy_2526 = (
    airport_costs["Annual_Gate_Cost_2526"].sum() * (4/12)
    + airport_costs["Estimated_2627"].sum() * (8/12)
)

print(f"FY2025-26 gate costs: ${gate_total_fy_2526:,.2f}")

FY2025-26 gate costs: $10,023,125.91


#### Flights

In [260]:
flights_2526 = flights_2526 = flight_costs[
    (flight_costs["Cost_Date"] >= "2025-09-01") &
    (flight_costs["Cost_Date"] < "2026-09-01")
]
flights_2526_total_costs = flights_2526["Total"].sum().round(2)
flights_2526_total_crew = flights_2526["Crew"].sum().round(2)
flights_2526_total_fuel = flights_2526["Fuel"].sum().round(2)
flights_2526_total_food = flights_2526["Food"].sum().round(2)
flights_2526_total_maintenance = flights_2526["Maintenance"].sum().round(2)

print(f"Total costs for 2024-2025 were: ${flights_2526_total_costs}")
print(f"Total crew costs for 2024-2025 were: ${flights_2526_total_crew}")
print(f"Total fuel costs for 2024-2025 were: ${flights_2526_total_fuel}")
print(f"Total food costs for 2024-2025 were: ${flights_2526_total_food}")
print(f"Total maintenance costs for 2024-2025 were: ${flights_2526_total_maintenance}")

Total costs for 2024-2025 were: $238376476.25
Total crew costs for 2024-2025 were: $74767150.35
Total fuel costs for 2024-2025 were: $83182330.22
Total food costs for 2024-2025 were: $9039515.73
Total maintenance costs for 2024-2025 were: $71387479.95


#### All Costs

In [261]:
total_costs_2526 = gate_total_fy_2526 + flights_2526_total_costs
print(f"Total costs for the fiscal year 2025-2026: ${total_costs_2526}")

Total costs for the fiscal year 2025-2026: $248399602.1578863


### Firm Revenue

In [262]:
bookings.head()

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
0,BK100000,2026,0.0,0.0,0.00,0.0,0.0,683.89,683.89,1,Direct
1,BK100001,2025,0.0,0.0,27.81,0.0,0.0,558.37,586.18,1,Direct
2,BK100002,2025,0.0,0.0,0.00,0.0,0.0,796.89,796.89,1,Direct
3,BK100003,2025,0.0,0.0,0.00,0.0,0.0,596.12,596.12,1,Direct
4,BK100004,2025,0.0,0.0,0.00,0.0,0.0,743.32,743.32,1,Direct


In [263]:
bookings_2526 = bookings[bookings["Fiscal_Year"] == 2026].copy()
bookings_2526.head()

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
0,BK100000,2026,0.0,0.0,0.0,0.0,0.00,683.89,683.89,1,Direct
5,BK100005,2026,0.0,0.0,0.0,0.0,0.00,1395.40,1395.40,1,Direct
8,BK100008,2026,0.0,0.0,0.0,0.0,0.00,562.55,562.55,1,Direct
10,BK100010,2026,0.0,0.0,0.0,0.0,0.00,1942.58,1942.58,2,Connecting
12,BK100012,2026,0.0,0.0,12.1,0.0,58.08,1051.87,1122.05,1,Direct


In [264]:
# 2024-2025 revenue streams

# total 
bookings_total_cost_2526 = bookings_2526["Total"].sum().round(2)
# change fee
change_fee_2526 = bookings_2526["Change Fee"].sum().round(2)
# extra baggage
baggage_2526 = bookings_2526["Extra Baggage"].sum().round(2)
# food and onboard
foodboard_2526 = bookings_2526["Food & Onboard"].sum().round(2)
# refund
refund_2526 = bookings_2526["Refund"].sum().round(2)
# seat upgrades
upgrades_2526 = bookings_2526["Seat Upgrade"].sum().round(2)
# ticket
ticket_2526 = bookings_2526["Ticket"].sum().round(2)

In [265]:
# total
print(f"Total revenue for 2026 fiscal year: ${bookings_total_cost_2526}")
# change
print(f"Total revenue from changing flights for 2026 fiscal year: ${change_fee_2526}")
# extra baggage
print(f"Total revenue from extra baggage for 2026 fiscal year: ${baggage_2526}")
# food
print(f"Total revenue from food for 2026 fiscal year: ${foodboard_2526}")
# refund
print(f"Total refunds for 2026 fiscal year: ${refund_2526}")
# upgrade
print(f"Total revenue from upgrades for 2026 fiscal year: ${upgrades_2526}")
# ticket
print(f"Total revenue from tickets for 2026 fiscal year: ${ticket_2526}")

Total revenue for 2026 fiscal year: $190163384.53
Total revenue from changing flights for 2026 fiscal year: $604750.0
Total revenue from extra baggage for 2026 fiscal year: $2141402.49
Total revenue from food for 2026 fiscal year: $494249.37
Total refunds for 2026 fiscal year: $-7846129.62
Total revenue from upgrades for 2026 fiscal year: $729899.26
Total revenue from tickets for 2026 fiscal year: $194039213.03


#### Connecting vs Direct

In [266]:
bookings_c2526 = bookings_2526[bookings_2526["Booking_Type"] == "Connecting"].copy()
bookings_d2526 = bookings_2526[bookings_2526["Booking_Type"] == "Direct"].copy()

In [267]:
bookings_c2526["Total"].sum()

np.float64(13242414.7)

In [268]:
bookings_d2526["Total"].sum()


np.float64(176920969.82999998)

In [269]:
print(f"There are {bookings_c2526.shape[0]} direct flights and {bookings_d2526.shape[0]} connecting")
prop_c2526 = (bookings_c2526.shape[0] / bookings_2526.shape[0]) * 100
prop_d2526 = (bookings_d2526.shape[0] / bookings_2526.shape[0]) * 100
print(f"Proportion of direct flights: {round(prop_d2526, 2)}%")
print(f"Proportion of connecting flights: {round(prop_c2526, 2)}%")

There are 6788 direct flights and 178122 connecting
Proportion of direct flights: 96.33%
Proportion of connecting flights: 3.67%


In [270]:
revenue_prop_c2526 = (bookings_c2526["Total"].sum() / bookings_2526["Total"].sum()) * 100
revenue_prop_d2526 = (bookings_d2526["Total"].sum() / bookings_2526["Total"].sum()) * 100

print(f"Proportion of revenue from direct flights: {round(revenue_prop_d2526, 2)}%")
print(f"Proportion of revenue connecting flights: {round(revenue_prop_c2526, 2)}%")

Proportion of revenue from direct flights: 93.04%
Proportion of revenue connecting flights: 6.96%


## Changes from 2024-2025 to 2025-2026

### Firm Costs

#### Gates

In [271]:
print(f"NorthByte increased spending on gates by: ${gate_total_fy_2526 - gate_total_fy_2425}")

print(f"Increasing their spending by on gates by: {(((gate_total_fy_2526 - gate_total_fy_2425)/gate_total_fy_2425) * 100).round(2)}%")

NorthByte increased spending on gates by: $216525.90788628906
Increasing their spending by on gates by: 2.21%


#### Flights

In [272]:
# flights
print(f"NorthByte increased spending on flights by: ${(flights_2526_total_costs - flights_2425_total_costs).round(2)}")
print(f"Increasing their spending by on flights by: {(((flights_2526_total_costs - flights_2425_total_costs)/flights_2425_total_costs) * 100).round(2)}%")

# crew
print(f"NorthByte increased spending on crew by: ${(flights_2526_total_crew - flights_2425_total_crew).round(2)}")
print(f"Increasing their spending by on crew by: {(((flights_2526_total_crew - flights_2425_total_crew)/flights_2425_total_crew) * 100).round(2)}%")
# fuel
print(f"NorthByte increased spending on fuel by: ${(flights_2526_total_fuel - flights_2425_total_fuel).round(2)}")
print(f"Increasing their spending by on fuel by: {(((flights_2526_total_fuel - flights_2425_total_fuel)/flights_2425_total_fuel) * 100).round(2)}%")
# food
print(f"NorthByte increased spending on food by: ${(flights_2526_total_food - flights_2425_total_food).round(2)}")
print(f"Increasing their spending by on food by: {(((flights_2526_total_food - flights_2425_total_food)/flights_2425_total_food) * 100).round(2)}%")
# maintenance
print(f"NorthByte increased spending on maintenance by: ${(flights_2526_total_maintenance - flights_2425_total_maintenance).round(2)}")
print(f"Increasing their spending by on maintenance by: {(((flights_2526_total_maintenance - flights_2425_total_maintenance)/flights_2425_total_maintenance) * 100).round(2)}%")


NorthByte increased spending on flights by: $45390912.26
Increasing their spending by on flights by: 23.52%
NorthByte increased spending on crew by: $15049420.86
Increasing their spending by on crew by: 25.2%
NorthByte increased spending on fuel by: $16121247.14
Increasing their spending by on fuel by: 24.04%
NorthByte increased spending on food by: $1216888.02
Increasing their spending by on food by: 15.56%
NorthByte increased spending on maintenance by: $13003356.24
Increasing their spending by on maintenance by: 22.27%


#### All Costs

In [273]:
print(f"Costs for the fiscal year 2024-2025 to 2025-2026 increased by a total of: ${(total_costs_2526 - total_costs_2425).round(2)}")
print(f"Increasing by: {(((total_costs_2526 - total_costs_2425)/total_costs_2425)*100).round(2)}%")


Costs for the fiscal year 2024-2025 to 2025-2026 increased by a total of: $45607438.17
Increasing by: 22.49%


#### Firm Revenue

### Visualizations

In [275]:
revenue_by_year = bookings.groupby("Fiscal_Year", as_index=False)["Total"].sum()

alt.Chart(revenue_by_year).mark_bar().encode(
    x="Fiscal_Year:O",
    y="Total:Q",
    color="Fiscal_Year:N"
)

alt.Chart(...)

In [277]:
financial_summary = pd.DataFrame({
    "Fiscal_Year": [2025, 2026],
    "Revenue": [
        bookings_total_cost_2425,
        bookings_total_cost_2526
    ],
    "Costs": [
        total_costs_2425,
        total_costs_2526
    ]
})

financial_long = financial_summary.melt(
    id_vars="Fiscal_Year",
    var_name="Category",
    value_name="Amount"
)

In [278]:
alt.Chart(financial_long).mark_bar().encode(
    x="Fiscal_Year:O",
    y="Amount:Q",
    color="Category:N",
    xOffset="Category:N",
    tooltip=["Fiscal_Year", "Category", "Amount"]
).properties(
    title="Revenue vs Costs by Fiscal Year"
)

alt.Chart(...)

In [279]:
financial_summary["Profit"] = (
    financial_summary["Revenue"] - financial_summary["Costs"]
)

financial_summary["Profit_Margin"] = (
    financial_summary["Profit"] / financial_summary["Revenue"] * 100
)

financial_summary

,Fiscal_Year,Revenue,Costs,Profit,Profit_Margin
0,2025,1.809080e+08,2.027922e+08,-2.188418e+07,-12.096854
1,2026,1.901634e+08,2.483996e+08,-5.823622e+07,-30.624306


In [280]:
revenue_categories = pd.DataFrame({
    "Fiscal_Year": [2025, 2026],
    "Ticket": [ticket_2425, ticket_2526],
    "Extra Baggage": [baggage_2425, baggage_2526],
    "Seat Upgrade": [upgrades_2425, upgrades_2526],
    "Change Fee": [change_fee_2425, change_fee_2526],
    "Food & Onboard": [foodboard_2425, foodboard_2526],
    "Refund": [refund_2425, refund_2526]
})

cost_categories = pd.DataFrame({
    "Fiscal_Year": [2025, 2026],
    "Fuel": [flights_2425_total_fuel, flights_2526_total_fuel],
    "Crew": [flights_2425_total_crew, flights_2526_total_crew],
    "Maintenance": [flights_2425_total_maintenance, flights_2526_total_maintenance],
    "Food": [flights_2425_total_food, flights_2526_total_food],
    "Gates": [gate_total_fy_2425, gate_total_fy_2526]
})

In [284]:
revenue_long = revenue_categories.melt(
    id_vars="Fiscal_Year",
    var_name="Category",
    value_name="Amount"
)

alt.Chart(revenue_long).mark_bar().encode(
    x="Category:N",
    y="Amount:Q",
    color="Fiscal_Year:N",
    xOffset="Fiscal_Year:N",
    tooltip=["Category", "Fiscal_Year", "Amount"]
).properties(
    title="Revenue by Category",
    width=600
)

alt.Chart(...)

In [293]:
revenue_change = revenue_categories.set_index("Fiscal_Year").T

# Calculate difference between years
revenue_change["Change"] = revenue_change[2026] - revenue_change[2025]

# Convert index into a Category column
revenue_change = revenue_change.reset_index().rename(
    columns={"index": "Category"}
)

# Plot only the necessary columns
alt.Chart(revenue_change[["Category", "Change"]]).mark_bar().encode(
    y=alt.Y("Category:N", sort="-x"),
    x=alt.X("Change:Q", axis=alt.Axis(format="~s")),
    color=alt.condition(
        alt.datum.Change >= 0,
        alt.value("#2A9D8F"),
        alt.value("#E76F51")
    ),
    tooltip=[
        "Category",
        alt.Tooltip("Change:Q", format="$,.0f")
    ]
).properties(
    title="Change in Revenue by Category",
    width=550,
    height=300
)

alt.Chart(...)

In [297]:
ancillary_change = revenue_change[
    revenue_change["Category"] != "Ticket"
]

alt.Chart(ancillary_change[["Category", "Change"]]).mark_bar().encode(
    y=alt.Y("Category:N", sort="-x", title=None),
    x=alt.X(
        "Change:Q",
        title="Change in Revenue ($)",
        axis=alt.Axis(format="~s")
    ),
    color=alt.condition(
        alt.datum.Change >= 0,
        alt.value("#2A9D8F"),
        alt.value("#E76F51")
    ),
    tooltip=[
        "Category",
        alt.Tooltip("Change:Q", format="$,.0f")
    ]
).properties(
    title="Change in Ancillary Revenue and Refunds",
    width=550,
    height=250
)

alt.Chart(...)

In [298]:
revenue_change["Percent_Change"] = (
    revenue_change["Change"] / revenue_change[2025].abs() * 100
)


In [287]:
cost_long = cost_categories.melt(
    id_vars="Fiscal_Year",
    var_name="Category",
    value_name="Amount"
)

alt.Chart(cost_long).mark_bar().encode(
    x="Category:N",
    y="Amount:Q",
    color="Fiscal_Year:N",
    xOffset="Fiscal_Year:N",
    tooltip=["Category", "Fiscal_Year", "Amount"]
).properties(
    title="Operating Costs by Category",
    width=600
)

alt.Chart(...)